# VideoMind Inference on Google Colab

A guided Colab notebook for launching the VideoMind demo with clear setup steps, optional custom video upload, and a default preloaded example so you can run it with no extra setup.


## What this notebook does

- Clones the VideoMind repository into Colab.
- Installs the runtime dependencies and video tools.
- Optionally logs in to Hugging Face if you use a private token.
- Lets you upload your own video in one cell.
- Launches the Gradio demo with a prefilled video, prompt, and roles.


## 1) Prepare the Colab runtime

Use a **GPU** runtime for best results. T4 works, but A100 is faster.


In [ ]:
%cd /content
!rm -rf VideoMind
!git clone https://github.com/selimq/VideoMind.git


In [ ]:
%cd /content/VideoMind
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt


In [ ]:
!apt-get -y install ffmpeg


## 2) Optional: sign in to Hugging Face

If model downloads fail, add `HF_TOKEN` in Colab Secrets and run this cell.


In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception as exc:
    token = None
    print(f'Could not read HF_TOKEN from Colab secrets: {exc}')

if token:
    login(token=token)
    print('Logged in to Hugging Face')
else:
    print('HF_TOKEN not found in Colab secrets; continuing without login')


## 3) Optional: upload your own video

Run this cell only if you want to demo a custom clip. If you skip it, the notebook will use a bundled VideoMind example automatically.


In [ ]:
from pathlib import Path

CUSTOM_VIDEO_PATH = None

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        CUSTOM_VIDEO_PATH = next(iter(uploaded))
        print(f'Using uploaded video: {CUSTOM_VIDEO_PATH}')
    else:
        print('No file uploaded; falling back to the bundled example video.')
except Exception as exc:
    print(f'Video upload skipped: {exc}')

if CUSTOM_VIDEO_PATH and not Path(CUSTOM_VIDEO_PATH).exists():
    raise FileNotFoundError(f'Uploaded video not found: {CUSTOM_VIDEO_PATH}')


## 4) Launch VideoMind

This cell preloads the demo with either your uploaded video or the first bundled example, so the interface opens ready to use.


In [ ]:
import os
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/VideoMind:' + os.environ.get('PYTHONPATH', '')

import demo.app as app

repo_root = str(Path(app.PATH).parent)

def pick_bundled_example():
    for example in getattr(app, 'EXAMPLES', []):
        if isinstance(example, (list, tuple)) and len(example) >= 3:
            return example[0], example[1], example[2]
    return None, 'Describe what is happening in this video.', ['pla', 'gnd', 'ver', 'ans']

if CUSTOM_VIDEO_PATH:
    default_video = CUSTOM_VIDEO_PATH
    default_prompt = 'Describe what is happening in this video.'
    default_role = ['pla', 'gnd', 'ver', 'ans']
else:
    default_video, default_prompt, default_role = pick_bundled_example()
    if default_video is None:
        raise RuntimeError('No bundled examples are available. Upload a custom video in the previous cell.')

print('Default video:', default_video)
print('Default prompt:', default_prompt)
print('Default roles:', default_role)

demo = app.build_demo(
    default_video=default_video,
    default_prompt=default_prompt,
    default_role=default_role,
)
demo.queue()
demo.launch(share=True, server_name='0.0.0.0', allowed_paths=[repo_root])


## 5) Tips

- The demo already includes a video upload component, so you can replace the default clip inside Gradio too.
- Keep the runtime alive while the demo is open.
- If the first run is slow, it is usually downloading the base model and adapters.
